***Import Libraries***

In [2]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix



KeyboardInterrupt



***Dataset Splitting***

***For this model 
we have our dataset called "data" in an path A/B/C/data
this dataset contain 2 classes "Nagative & Positive" and for this model we need to create 
3 other folders called "train" , "validates" and "test" and we have to put in each folder the 2 classes Nagative & Positive***

In [ ]:
import os
import shutil
import random

# Path to your dataset
base_dir = r"C:\Users\hp\anaconda3\Share Folder For Work\anaconda3\Projects work on it 2026\Concret Cracks\data"

# Output directories
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "validate")
test_dir  = os.path.join(base_dir, "test")

# Classes
classes = ["Negative", "Positive"]

# Create train/validate/test folders
for split_dir in [train_dir, val_dir, test_dir]:
    os.makedirs(split_dir, exist_ok=True)
    for cls in classes:
        os.makedirs(os.path.join(split_dir, cls), exist_ok=True)

# Split ratios
train_split = 0.7
val_split   = 0.15
test_split  = 0.15

# Function to split and copy files
def split_data(class_name):
    src_dir = os.path.join(base_dir, class_name)
    files = os.listdir(src_dir)
    random.shuffle(files)

    total = len(files)
    train_end = int(train_split * total)
    val_end   = int((train_split + val_split) * total)

    train_files = files[:train_end]
    val_files   = files[train_end:val_end]
    test_files  = files[val_end:]

    for f in train_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(train_dir, class_name, f))
    for f in val_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(val_dir, class_name, f))
    for f in test_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(test_dir, class_name, f))

# Run for both classes
for cls in classes:
    split_data(cls)

print("Dataset successfully split into train/validate/test folders!")

***Assign The Path for Each Folder from the 3 that we created above***

In [ ]:
# Paths to your dataset (adjust if needed)
# C:\Users\hp\anaconda3\Share Folder For Work\anaconda3\Projects work on it 2026\Concret Cracks
train_dir = "anaconda3/dataset/train"
train_dir = r"C:\Users\hp\anaconda3\Share Folder For Work\anaconda3\Projects work on it 2026\Concret Cracks\data\train"
val_dir   = r"C:\Users\hp\anaconda3\Share Folder For Work\anaconda3\Projects work on it 2026\Concret Cracks\data\validate"
test_dir  = r"C:\Users\hp\anaconda3\Share Folder For Work\anaconda3\Projects work on it 2026\Concret Cracks\data\test"


***Image Parameters***

In [ ]:
# Image parameters
img_height, img_width = 224,224
batch_size = 32


***Data generators (with normalization)***

In [ ]:
# Data generators (with normalization)
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)



In [ ]:
train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(img_height, img_width),
    batch_size=batch_size, class_mode='binary'
)

val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=(img_height, img_width),
    batch_size=batch_size, class_mode='binary'
)

test_gen = test_datagen.flow_from_directory(
    test_dir, target_size=(img_height, img_width),
    batch_size=batch_size, class_mode='binary',
    shuffle=False
)


***Show some sample images before normalization***

In [ ]:
# Show some sample images correctly
sample_images, sample_labels = next(train_gen)

plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(sample_images[i])  # no astype("uint8")
    plt.title("Label: " + str(int(sample_labels[i])))
    plt.axis("off")
plt.show()

***CNN Model Structure***

In [ ]:
# Build CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same',input_shape=(img_height, img_width, 3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu', padding='same',),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu', padding='same',),
    MaxPooling2D(2,2),
    
    #Conv2D(128, (3,3), activation='relu', padding='same',),
    #MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu', padding='same',),
    MaxPooling2D(2,2),
    
    Conv2D(512, (3,3), activation='relu', padding='same',),
    MaxPooling2D(2,2),
    
    #Dense(128, activation='relu'),
    #Dropout(0.5),
    Flatten(),
    Dense(128, activation='relu'),
    #Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


***Model Fitting***

In [ ]:
# Callbacks
checkpoint = ModelCheckpoint("cnn_model.h5", save_best_only=True, monitor="val_accuracy", mode="max")
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# Train model
history = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=[checkpoint, early_stop]
)



***Normalize Images***

In [ ]:
# Plot training history
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()


***Improve Performance***

In [ ]:
# Plot training vs validation loss
plt.figure(figsize=(8,6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_gen)
print("Test Accuracy:", test_acc)



***Build CNN Model***

In [ ]:
# Confusion matrix
y_pred = (model.predict(test_gen) > 0.5).astype("int32")
cm = confusion_matrix(test_gen.classes, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

print(classification_report(test_gen.classes, y_pred, target_names=['Negative','Positive']))



In [ ]:
# Save model
model.save("cnn_concrete_crack.h5")



***Compile Model***

In [ ]:
# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("tflite_b_concrete.tflite", "wb") as f:
    f.write(tflite_model)


In [ ]:
from sklearn.metrics import roc_curve, auc

# Get predicted probabilities (not thresholded)
y_pred_prob = model.predict(test_gen)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(test_gen.classes, y_pred_prob)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

print("AUC Score:", roc_auc)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))

***Save the Final Model***

In [ ]:
model.save("cnn_crack_model.h5")